# Restaurant Footfall Forecasting

## Project Overview

Forecasts **daily restaurant customer count** over a 14-day horizon. Synthetic data spans 2 years with weekly dining patterns, seasonal variation, and holiday spikes.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily customer counts, predict the next 14 days. Restaurants need footfall forecasts for staffing, food procurement, and table management.

## Why This Project Matters

Food waste costs the restaurant industry billions annually. Accurate footfall forecasts enable optimal food ordering, reduce waste by 20-30%, and ensure adequate staffing for service quality. Under-staffing loses revenue; over-staffing erodes margins.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily customer counts
- Strong weekly pattern (Friday/Saturday peaks, Monday/Tuesday lows)
- Seasonal variation (summer dining out, winter dips)
- Holiday spikes (Valentine's Day, Mother's Day, Christmas)
- Growth trend (restaurant popularity)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `customers` | Daily customer count |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'customers'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 200 + 0.05 * t
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow == 0: weekly[i] = -40  # Monday
    elif dow == 1: weekly[i] = -30  # Tuesday
    elif dow <= 3: weekly[i] = 10  # Wed-Thu
    elif dow == 4: weekly[i] = 50  # Friday
    elif dow == 5: weekly[i] = 70  # Saturday peak
    else: weekly[i] = 30  # Sunday brunch

seasonal = 25 * np.sin(2 * np.pi * (t - 90) / 365)

holiday = np.zeros(N_POINTS)
for i in range(N_POINTS):
    m, d = dates[i].month, dates[i].day
    if m == 2 and d == 14: holiday[i] = 80  # Valentine's
    elif m == 5 and 8 <= d <= 14 and dates[i].dayofweek == 6: holiday[i] = 60  # Mother's Day
    elif m == 12 and 24 <= d <= 25: holiday[i] = -50  # Christmas (closed/reduced)
    elif m == 12 and d == 31: holiday[i] = 100  # NYE

noise = np.random.normal(0, 15, N_POINTS)
customers = base + weekly + seasonal + holiday + noise
customers = np.maximum(customers, 30).round(0).astype(int)

df = pd.DataFrame({'date': dates, 'customers': customers})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,customers
0,2022-01-01,252
1,2022-01-02,203
2,2022-01-03,145
3,2022-01-04,168
4,2022-01-05,182


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('customers Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("customers Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

customers Statistics:
count    730.000000
mean     232.769863
std       45.879297
min      113.000000
25%      198.000000
50%      235.000000
75%      266.000000
max      374.000000
Name: customers, dtype: float64

CV: 0.197
Skewness: -0.043


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:       50.9 | RMSE:       62.2 | MAPE: 27.49%
Seasonal Naive (7)             | MAE:       26.6 | RMSE:       50.8 | MAPE: 12.07%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                          Adjusted R-Squared  R-Squared        RMSE  Time Taken
Model                                                                          
KernelRidge                       413.325521 -30.717348  237.038224    0.029986
MLPRegressor                       39.468234  -1.959095   72.401806    0.274600
GaussianProcessRegressor           30.437728  -1.264441   63.335970    0.042898
QuantileRegressor                  15.635460  -0.125805   44.658226    0.049985
DummyRegressor                     14.877647  -0.067511   43.486673    0.006127
DecisionTreeRegressor               8.738904   0.404700   32.474166    0.012149
ExtraTreeRegressor                  6.722931   0.559775   27.925922    0.008892
NuSVR                               5.801406   0.630661   25.578936    0.027304
RANSACRegressor                     5.438612   0.658568   24.593587    0.063337
SVR                                 5.241989   0.673693   24.042689    0.022317


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: extra_tree
FLAML (extra_tree)             | MAE:       16.0 | RMSE:       18.8 | MAPE:  7.52%


FLAML Test (extra_tree)        | MAE:       19.0 | RMSE:       32.6 | MAPE:  9.40%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:       19.8 | RMSE:       33.2 | MAPE:  9.34%
SF AutoETS                     | MAE:       19.0 | RMSE:       32.8 | MAPE:  8.90%
SF SeasonalNaive               | MAE:       21.9 | RMSE:       36.3 | MAPE: 11.45%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                  model       MAE      RMSE      MAPE
     FLAML (extra_tree) 15.987347 18.832456  7.518823
FLAML Test (extra_tree) 19.000605 32.599327  9.397294
             SF AutoETS 19.038181 32.818030  8.903358
           SF AutoARIMA 19.809988 33.207491  9.343199
       SF SeasonalNaive 21.928572 36.250123 11.447455
     Seasonal Naive (7) 26.642857 50.783152 12.068536
     Naive (Last Value) 50.928571 62.230562 27.494006

Top 3:
                  model       MAE      RMSE     MAPE
     FLAML (extra_tree) 15.987347 18.832456 7.518823
FLAML Test (extra_tree) 19.000605 32.599327 9.397294
             SF AutoETS 19.038181 32.818030 8.903358


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -3.63, Std: 32.40


## Interpretation and Business Insight

- Friday and Saturday are peak dining days (30-40% above baseline)
- Monday/Tuesday are consistently lowest
- Valentine's Day and NYE create the biggest spikes
- Christmas may reduce footfall (closures or family dining at home)
- Summer has slightly higher footfall than winter

## Limitations

1. Synthetic — real footfall depends on reviews, promotions, weather
2. No weather features (rain reduces walk-in traffic)
3. No reservation data
4. No price or menu change effects
5. Single restaurant — chain forecasting needs store-level models

## How to Improve This Project

1. Add weather data (rain, temperature)
2. Include reservation counts as leading indicator
3. Add promotion and event calendar
4. Model lunch vs dinner separately
5. Use review sentiment as demand signal

## Production Considerations

- Daily footfall forecast for 1-2 weeks ahead
- Integration with staff scheduling systems
- Automated food procurement ordering
- Table reservation capacity management

## Common Mistakes

1. Ignoring weather — rain can drop walk-in traffic 20-30%
2. Not handling special days (Valentine's, NYE) separately
3. Using weekly averages instead of day-specific forecasts
4. Not distinguishing dine-in from takeout/delivery

## Mini Challenge / Exercises

1. Add a synthetic rain indicator and measure footfall impact
2. Model lunch vs dinner traffic separately
3. Build a food waste predictor from footfall forecast errors
4. Detect anomalous days (unexplained surges or drops)

## Final Summary / Key Takeaways

1. Restaurant footfall has strong weekly patterns (weekend peaks)
2. Special dining holidays create predictable spikes
3. Weather is the key missing short-term predictor
4. Accurate forecasts directly reduce food waste and improve staffing
5. Combining ML with statistical models captures both patterns and trends

In [18]:
import json
metrics = {
    'project': 'Restaurant Footfall Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Restaurant Footfall Forecasting — Complete ===")

Metrics saved.

=== Restaurant Footfall Forecasting — Complete ===
